# Entrenamiento del modelo de tiempo de entrega

Notebook operativo ejecutado por Papermill desde `train.sh`. Entrena un pipeline Spark con el nuevo dataset, calcula una partición temporal, compara contra la media y persiste el modelo completo para inferencia en streaming.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.sql.functions import lit

DATA = Path('/home/jovyan/shared_data/food_delivery/train.csv')
if not DATA.exists():
    os.environ.setdefault('KAGGLEHUB_CACHE', '/home/jovyan/logs/kagglehub')
    import kagglehub
    DATA = Path(kagglehub.dataset_download('gauravmalik26/food-delivery-dataset')) / 'train.csv'
if not DATA.exists():
    raise FileNotFoundError(f'No se ha encontrado train.csv en {DATA}')
MODEL = '/home/jovyan/Food_delivery/models/pipeline_model.bin'
SEED = 42


In [ ]:
prepared_pdf = pd.read_csv(DATA, skipinitialspace=True)
if len(prepared_pdf) != 45593:
    raise RuntimeError(f'El dataset de Kaggle ha cambiado: se esperaban 45593 filas y hay {len(prepared_pdf)}')
for column in prepared_pdf.select_dtypes('object'):
    prepared_pdf[column] = prepared_pdf[column].str.strip().replace({'NaN': np.nan, 'nan': np.nan, 'conditions NaN': np.nan})
prepared_pdf['delivery_time_min'] = pd.to_numeric(prepared_pdf['Time_taken(min)'].str.extract(r'(\d+)', expand=False), errors='coerce')
prepared_pdf['order_date'] = pd.to_datetime(prepared_pdf['Order_Date'], dayfirst=True, errors='coerce')
prepared_pdf['hour_of_day'] = pd.to_numeric(prepared_pdf['Time_Orderd'].str.extract(r'^(\d+)')[0], errors='coerce').mod(24)
prepared_pdf = prepared_pdf.rename(columns={'Delivery_person_Age':'delivery_person_age','Delivery_person_Ratings':'delivery_person_ratings','Restaurant_latitude':'restaurant_latitude','Restaurant_longitude':'restaurant_longitude','Delivery_location_latitude':'delivery_location_latitude','Delivery_location_longitude':'delivery_location_longitude','Weatherconditions':'weather_conditions','Road_traffic_density':'road_traffic_density','Vehicle_condition':'vehicle_condition','Type_of_order':'type_of_order','Type_of_vehicle':'type_of_vehicle','Festival':'festival','City':'city'})
coordinates = ['restaurant_latitude','restaurant_longitude','delivery_location_latitude','delivery_location_longitude']
for column in coordinates:
    prepared_pdf[column] = pd.to_numeric(prepared_pdf[column], errors='coerce').abs()
for column in ['delivery_person_age','delivery_person_ratings','vehicle_condition','multiple_deliveries']:
    prepared_pdf[column] = pd.to_numeric(prepared_pdf[column], errors='coerce')
prepared_pdf = prepared_pdf.loc[prepared_pdf.delivery_time_min.notna() & prepared_pdf.order_date.notna() & prepared_pdf[coordinates].notna().all(axis=1) & prepared_pdf[['restaurant_latitude','restaurant_longitude']].ne(0).all(axis=1)].copy()
lat1, lon1 = np.radians(prepared_pdf.restaurant_latitude), np.radians(prepared_pdf.restaurant_longitude)
lat2, lon2 = np.radians(prepared_pdf.delivery_location_latitude), np.radians(prepared_pdf.delivery_location_longitude)
haversine_a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
prepared_pdf['distance_km'] = 2 * 6371.0 * np.arcsin(np.sqrt(haversine_a))
prepared_pdf['day_of_week'] = prepared_pdf.order_date.dt.dayofweek + 1
prepared_pdf['hour_sin'] = np.sin(2*np.pi*prepared_pdf.hour_of_day/24)
prepared_pdf['hour_cos'] = np.cos(2*np.pi*prepared_pdf.hour_of_day/24)
prepared_pdf['weather_conditions'] = prepared_pdf.weather_conditions.str.replace(r'^conditions\s+', '', regex=True)
categorical_cols = ['weather_conditions','road_traffic_density','type_of_order','type_of_vehicle','festival','city']
for column in categorical_cols:
    prepared_pdf[column] = prepared_pdf[column].fillna('Unknown').astype(str).str.strip()
temporal_cutoff = prepared_pdf.order_date.quantile(0.8)
prepared_pdf['dataset_split'] = np.where(prepared_pdf.order_date <= temporal_cutoff, 'train', 'test')
numeric_cols = ['delivery_person_age','delivery_person_ratings','vehicle_condition','multiple_deliveries','distance_km','hour_sin','hour_cos','day_of_week']
target = 'delivery_time_min'
prepared_pdf = prepared_pdf[numeric_cols + categorical_cols + [target, 'dataset_split']]


In [ ]:
spark = (SparkSession.builder.appName('Train-Food-Delivery-Time-RF')
    .master('spark://agile:7077').config('spark.executor.memory','2g')
    .config('spark.driver.memory','2g').getOrCreate())
spark.sparkContext.setLogLevel('WARN')

prepared = spark.createDataFrame(prepared_pdf)
train_data = prepared.filter("dataset_split = 'train'").drop('dataset_split')
test_data = prepared.filter("dataset_split = 'test'").drop('dataset_split')
train_count, test_count = train_data.count(), test_data.count()


In [ ]:
imputed_cols = [c + '_imp' for c in numeric_cols]
imputer = Imputer(inputCols=numeric_cols, outputCols=imputed_cols, strategy='median')
indexers = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_vec', handleInvalid='keep') for c in categorical_cols]
assembler = VectorAssembler(inputCols=imputed_cols + [c+'_vec' for c in categorical_cols], outputCol='features', handleInvalid='keep')
regressor = RandomForestRegressor(labelCol=target, featuresCol='features', numTrees=100, maxDepth=10, minInstancesPerNode=3, featureSubsetStrategy='0.8', seed=SEED)
pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler, regressor])
model = pipeline.fit(train_data)
predictions = model.transform(test_data)
rmse_eval = RegressionEvaluator(labelCol=target,predictionCol='prediction',metricName='rmse')
mae_eval = RegressionEvaluator(labelCol=target,predictionCol='prediction',metricName='mae')
r2_eval = RegressionEvaluator(labelCol=target,predictionCol='prediction',metricName='r2')
rmse, mae, r2 = rmse_eval.evaluate(predictions), mae_eval.evaluate(predictions), r2_eval.evaluate(predictions)
train_mean = train_data.selectExpr(f'avg({target}) AS mean_target').first()['mean_target']
baseline = test_data.withColumn('prediction',lit(float(train_mean)))
baseline_rmse = rmse_eval.evaluate(baseline)
print(f'RandomForest RMSE={rmse:.3f}, MAE={mae:.3f}, R2={r2:.3f}; media RMSE={baseline_rmse:.3f}')
if rmse >= baseline_rmse:
    raise RuntimeError('El modelo no supera la referencia de la media')


In [ ]:
model.write().overwrite().save(MODEL)
print(f'Filas limpias={len(prepared_pdf)}, train={train_count}, test={test_count}, corte temporal={temporal_cutoff.date()}')
predictions.select(target,'prediction','distance_km','road_traffic_density').show(10,truncate=False)
spark.stop()
